In [ ]:
# Cell 1: restore the uploaded project payload into a writable workspace.
from pathlib import Path
import shutil
import sys
import subprocess

inputs = Path("/kaggle/input")
payload_candidates = sorted(inputs.rglob("final_eval"))
payload_candidates = [p for p in payload_candidates if p.is_dir()]

if not payload_candidates:
    raise FileNotFoundError(
        "Could not find final_eval anywhere under /kaggle/input. "
        "Check that the symbiote-old-v2-comparison dataset is attached."
    )

# The payload currently has /kaggle/input/<dataset>/kaggle_old_v2_payload/final_eval.
payload_root = payload_candidates[0].parent
root = Path("/kaggle/working/symbiote")

if root.exists():
    shutil.rmtree(root)

shutil.copytree(payload_root, root)

print("Payload source:", payload_root)
print("Working directory:", root)
print("GPU check:")
subprocess.run(
    [sys.executable, "-c",
     "import torch; print(torch.cuda.is_available()); "
     "print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No CUDA')"],
    check=True,
)


In [ ]:
# Cell 2: install project dependencies and the spaCy pipeline.
from pathlib import Path
import subprocess
import sys

root = Path("/kaggle/working/symbiote")
root.mkdir(parents=True, exist_ok=True)

# The uploaded payload omitted the repository-level requirements.txt. Create it
# here so final_eval/requirements.txt can resolve its relative include.
root_requirements = """\
datasets
huggingface_hub
pandas
spacy
requests
scikit-learn
transformers
torch
python-dotenv
matplotlib
seaborn
ipykernel
sacrebleu
"""
requirements_path = root / "requirements.txt"
if not requirements_path.exists():
    requirements_path.write_text(root_requirements, encoding="utf-8")
    print("Created:", requirements_path)
else:
    print("Using existing:", requirements_path)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "final_eval/requirements.txt"],
    cwd=root,
    check=True,
)

# The compressor calls spacy.load("en_core_web_sm"); install that model in
# the notebook rather than assuming it is present in the Kaggle image.
subprocess.run(
    [sys.executable, "-m", "spacy", "download", "en_core_web_sm"],
    check=True,
)

print("Dependencies and spaCy model installed successfully.")


In [ ]:
# Cell 3
from pathlib import Path
import json
import pandas as pd
import torch

root = Path("/kaggle/working/symbiote")

cohort_path = root / "outputs/followup/old_v2/common_cohort.csv"
labels_path = root / "data/consensus_dataset.csv"
old_model = root / "models/span_scorer_historical"
v2_model = root / "models/span_scorer_v2_20261021"

assert cohort_path.is_file(), cohort_path
assert labels_path.is_file(), labels_path
assert (old_model / "split.json").is_file()
assert (v2_model / "split.json").is_file()
assert len(list(old_model.glob("fold*/scorer.pt"))) == 5
assert len(list(v2_model.glob("fold*/scorer.pt"))) == 5
assert torch.cuda.is_available()

cohort = pd.read_csv(cohort_path, keep_default_na=False)
old_split = json.loads((old_model / "split.json").read_text())
v2_split = json.loads((v2_model / "split.json").read_text())

print({
    "questions": len(cohort),
    "families": cohort["base_group_id"].nunique(),
    "old_split_questions": len(old_split["folds"]),
    "v2_split_questions": len(v2_split["folds"]),
    "methods": ["ours", "grouped_hybrid"],
    "ratios": [0.50, 0.75],
    "gpu": torch.cuda.get_device_name(0),
})

In [ ]:
# Cell 4
from pathlib import Path
import subprocess
import sys

root = Path("/kaggle/working/symbiote")
work = root / "outputs/followup/old_v2/v2"

command = [
    sys.executable,
    "-m",
    "final_eval.batch",
    "--cohort",
    "outputs/followup/old_v2/common_cohort.csv",
    "--work-dir",
    "outputs/followup/old_v2/v2",
    "--model-dir",
    "models/span_scorer_v2_20261021",
    "--labels-file",
    "data/consensus_dataset.csv",
    "--methods",
    "ours",
    "grouped_hybrid",
    "--ratios",
    ".50",
    ".75",
]

subprocess.run(command, cwd=root, check=True)

print("v2 compression completed.")
print("Output:", work / "compressed.csv")

In [ ]:
# Cell 5
from pathlib import Path
import subprocess
import sys

root = Path("/kaggle/working/symbiote")
work = root / "outputs/followup/old_v2/historical"

command = [
    sys.executable,
    "-m",
    "final_eval.batch",
    "--cohort",
    "outputs/followup/old_v2/common_cohort.csv",
    "--work-dir",
    "outputs/followup/old_v2/historical",
    "--model-dir",
    "models/span_scorer_historical",
    "--labels-file",
    "data/consensus_dataset.csv",
    "--methods",
    "ours",
    "grouped_hybrid",
    "--ratios",
    ".50",
    ".75",
    "--allow-legacy-checkpoint",
]

subprocess.run(command, cwd=root, check=True)

print("Historical compression completed.")
print("Output:", work / "compressed.csv")

In [ ]:
# Cell 6
from pathlib import Path
import pandas as pd

root = Path("/kaggle/working/symbiote")

for name in ["v2", "historical"]:
    path = root / f"outputs/followup/old_v2/{name}/compressed.csv"
    frame = pd.read_csv(path, keep_default_na=False)

    print("\n", name)
    print("rows:", len(frame))
    print("questions:", frame.uid.nunique())
    print("families:", frame.base_group_id.nunique())
    print("methods:", sorted(frame.method.unique()))
    print("ratios:", sorted(frame.target_ratio.unique()))
    print("statuses:", frame.status.value_counts().to_dict())
    print("budget:", frame.budget_compliant.value_counts().to_dict())

    assert len(frame) == 2956
    assert frame.uid.nunique() == 739
    assert set(frame.method) == {"ours", "grouped_hybrid"}
    assert set(frame.target_ratio) == {0.5, 0.75}
    assert frame.status.eq("ok").all()
    assert frame.budget_compliant.astype(bool).all()

In [ ]:
# Cell 7
from pathlib import Path
import shutil

root = Path("/kaggle/working/symbiote")
export_dir = Path("/kaggle/working/old_v2_comparison_results")

if export_dir.exists():
    shutil.rmtree(export_dir)

export_dir.mkdir(parents=True)

for name in ["v2", "historical"]:
    source = root / f"outputs/followup/old_v2/{name}"
    target = export_dir / name
    shutil.copytree(source, target)

shutil.copy2(
    root / "outputs/followup/old_v2/common_cohort.manifest.json",
    export_dir / "common_cohort.manifest.json",
)

archive = shutil.make_archive(
    "/kaggle/working/old_v2_comparison_results",
    "zip",
    export_dir,
)

print("Download this file:")
print(archive)